# QLoRA — Instruction Tuning an 8B Model in 4-Bit

**Base model:** [`unsloth/Meta-Llama-3.1-8B`](https://huggingface.co/unsloth/Meta-Llama-3.1-8B),
or [`Qwen/Qwen2.5-7B`](https://huggingface.co/Qwen/Qwen2.5-7B) — one switch away
**Data:** [`qiaojin/PubMedQA`](https://huggingface.co/datasets/qiaojin/PubMedQA) `pqa_labeled` — the
same 800 training records and the same 200 held-out records notebook 02 uses
**Hardware:** Colab **A100 or L4** · **Runtime:** ~60 minutes on an A100

---

## What this notebook does

Notebooks 01-03 are a pipeline: adapt to a domain, install instruction following, tune on
preferences — merging the adapter into the base weights between each stage. **This notebook is not
stage 4 of that pipeline.** It is a parallel, single-stage track that asks a different question:

> What changes when you stop holding the base model in fp16, hold it in **4 bits** instead, and
> spend the memory you saved on a model **six times larger**?

That is QLoRA. It is the technique most fine-tuning work reaches for once the model stops being
small enough to load whole, and it is the reason a 7B or 8B fine-tune fits on one rented GPU
instead of a node.

```
unsloth/Meta-Llama-3.1-8B   (base, no instruction tuning)
      │   16.1 GB in fp16  ->  ~5.7 GB in 4-bit NF4
      │
      │   one LoRA, r=16, on the same 800 PubMedQA records notebook 02 used
      ▼
   an 8B model that answers yes/no/maybe with a justification — and stops
```

## Why single-stage, and why that is not a dodge

Notebook 02's architecture is `base -> attach LoRA #1 -> merge -> attach LoRA #2`. **The merge is
mandatory there**, and merging into 4-bit weights is exactly the operation this repo has been
arguing against since `docs/concepts.md` §3. Training one adapter on the raw 4-bit base needs no
merge at all, so the architecture and the quantization stop fighting each other.

The merge does not disappear, though — it gets **measured**. Section 14 quantifies the
requantization error against the size of the update that was actually trained, then merges anyway
and re-scores, so the constraint this repo keeps citing finally has a number attached to it.

## What is held fixed, so the comparison means something

Identical to notebook 02: the 800/200 stratified split, `SEED`, `EVAL_FRACTION`, the prompt
template, completion-only masking, the grading regex, greedy decoding, and LoRA `r=16` /
`alpha=32` / the same seven `target_modules`. The effective batch is 8 in both, so the optimizer
takes the same number of steps.

Changed, deliberately and only: model scale, 4-bit NF4 base weights,
`prepare_model_for_kbit_training` in place of the hand-rolled fp32 upcast, a paged 8-bit
optimizer, and gradient checkpointing.

## What to expect

Three scored systems on the same 200 held-out articles: the 4-bit base with the adapter switched
off, the same model with QLoRA, and the same model again after the adapter has been merged into
the 4-bit weights. Plus a peak-memory figure, a weight-level measurement of what NF4 costs, and a
paired significance test on whether merging did detectable damage.

**Set your runtime to A100 or L4 first:** Runtime → Change runtime type → A100 GPU.
This notebook asserts an Ampere-or-later GPU and **will not run on a T4** — see section 1.

## 1. Runtime and dependencies

One extra pin over notebooks 01 and 02: `bitsandbytes`, which supplies the 4-bit linear layers and
the paged optimizer. Everything else is the same pin set.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or
      "No GPU found — set Runtime > Change runtime type > T4 GPU")

In [ ]:
# Pinned to match requirements-colab.txt. torch is deliberately left alone:
# Colab ships a CUDA-matched build and replacing it is slow and fragile.
%pip install -q "transformers==5.15.0" "peft==0.20.0" "datasets==5.0.1" "accelerate==1.14.0" "bitsandbytes==0.50.2"
# Colab preinstalls torchao 0.10.x. peft's optional torchao integration RAISES rather than
# degrading gracefully when it finds a version below its 0.16.0 minimum, and the check fires
# deep inside get_peft_model(). Nothing here uses torchao, so remove it rather than chase a
# compatible build against Colab's torch.
%pip uninstall -q -y torchao
print("\nRestart the runtime if Colab asks you to, then run this cell again and continue.")

### This notebook does not run on a T4

Notebooks 01-03 all branch on `SUPPORTS_BF16` and fall back to fp16 on a Turing T4. This one
asserts instead, because with 4-bit weights the fallback is not merely slower — it is slow enough
to change what you would choose to do.

`bnb_4bit_compute_dtype` is the precision every matmul runs in after the weights are dequantized on
the fly. On Ampere and later that is native bf16. On a T4 there is no bf16 unit, so the compute
dtype has to be fp16, and the dequantize-then-matmul path ends up several times slower on top of
the extra work 4-bit already costs. An 8B run that takes an hour on an A100 takes most of an
afternoon on a T4.

The free-tier promise in this repo's README covers notebooks 01-03. It does not cover this one.

In [ ]:
import gc, json, math, random, re
from pathlib import Path
from collections import Counter

import torch
import transformers
import peft
import bitsandbytes as bnb
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No CUDA device. Runtime > Change runtime type > A100 GPU."

# Notebooks 01-03 branch here and fall back to fp16. This one refuses, because 4-bit compute
# on a GPU with no native bfloat16 is slow enough to change the shape of the exercise.
CAPABILITY = torch.cuda.get_device_capability()
assert CAPABILITY[0] >= 8, (
    f"This notebook needs an Ampere-or-later GPU (sm_80+); found "
    f"sm_{CAPABILITY[0]}{CAPABILITY[1]} ({torch.cuda.get_device_name(0)}). "
    "A T4 is Turing (sm_75) and has no native bfloat16, so 4-bit compute would fall back to "
    "fp16 and run several times slower.\n"
    "Runtime > Change runtime type > A100 GPU (or L4)."
)
DTYPE = torch.bfloat16
TF_MAJOR = int(transformers.__version__.split(".")[0])
DTYPE_KW = "dtype" if TF_MAJOR >= 5 else "torch_dtype"   # transformers v5 renamed torch_dtype

TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"transformers {transformers.__version__} | peft {peft.__version__} | bitsandbytes {bnb.__version__}")
print(f"GPU {torch.cuda.get_device_name(0)}  sm_{CAPABILITY[0]}{CAPABILITY[1]}  {TOTAL_VRAM:.1f} GB")
print(f"compute dtype {DTYPE} | from_pretrained keyword {DTYPE_KW!r}")

# Fail fast on the torchao clash, rather than eight cells from now inside get_peft_model().
import importlib.metadata as _md


def _ver(s):
    return tuple(int(p) for p in s.split("+")[0].split(".")[:3] if p.isdigit())


try:
    _ta = _md.version("torchao")
    if _ver(_ta) < (0, 16, 0):
        raise RuntimeError(
            f"torchao {_ta} is too old for peft {peft.__version__} (needs >= 0.16.0).\n"
            "Nothing in this notebook uses torchao. Fix it with:\n"
            "    !pip uninstall -y torchao\n"
            "then Runtime > Restart session, then Runtime > Run all."
        )
    print(f"torchao      {_ta} (compatible)")
except _md.PackageNotFoundError:
    print("torchao      absent - fine, nothing here uses it")

## 2. Choosing the base model

Two profiles, both ungated, both **base** models rather than `-Instruct` — the point of the whole
repo is watching instruction following get installed, so starting from a model that already has it
would give away the answer.

| | `unsloth/Meta-Llama-3.1-8B` | `Qwen/Qwen2.5-7B` |
|---|---|---|
| parameters | 8.03B | 7.62B |
| layers / hidden | 32 / 4096 | 28 / 3584 |
| vocabulary | 128,256 | 152,064 |
| tokenizer | **same as `Llama-3.2-1B`** | different BPE |
| `pad_token` | `<\|finetune_right_pad_id\|>` | **collides with `eos_token`** |

The Llama profile is the reference run, for one specific reason: Llama 3.1 and Llama 3.2 share a
tokenizer, so notebook 02's `MAX_LEN=896` budget and its token-length statistics carry over
unchanged. The only things that differ between the two notebooks are then scale and quantization,
which is what makes the comparison worth printing.

**The Qwen gotcha, named.** Qwen-2.5 ships `pad_token == eos_token == <|endoftext|>`. Leave that
alone and every padded position in a batch is an EOS. In notebook 02's collator the padding is
masked to `-100` so it is not scored — but the `assert` in section 7 exists precisely because that
guarantee is one careless edit away from vanishing, and because generation pads with the same
token it stops on. `<|fim_pad|>` is already in Qwen's vocabulary, so switching to it costs nothing
and needs no embedding resize. Qwen also carries biases on `q/k/v_proj`; `bias="none"` leaves them
frozen, which is what we want.

In [ ]:
MODEL_PROFILES = {
    "llama-3.1-8b": {
        "model_id": "unsloth/Meta-Llama-3.1-8B",  # ungated mirror, same pattern as the 1B
        "max_len": 896,      # same tokenizer as Llama-3.2-1B, so notebook 02's budget carries over
        "pad_token": None,   # the mirror already sets <|finetune_right_pad_id|>, distinct from EOS
        "probe_layer": 8,
        "short": "llama31-8b",
    },
    "qwen2.5-7b": {
        "model_id": "Qwen/Qwen2.5-7B",
        "max_len": 1024,     # different BPE - section 7 re-measures and asserts no truncation
        "pad_token": "<|fim_pad|>",   # ships pad == eos; see the note above
        "probe_layer": 8,
        "short": "qwen25-7b",
    },
}

MODEL_KEY = "llama-3.1-8b"          # <-- the switch
PROFILE = MODEL_PROFILES[MODEL_KEY]
MODEL_ID = PROFILE["model_id"]
MODEL_SHORT = PROFILE["short"]

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"          # training, not generation
if PROFILE["pad_token"] is not None:
    tokenizer.pad_token = PROFILE["pad_token"]

assert tokenizer.pad_token_id != tokenizer.eos_token_id, (
    f"PAD and EOS must stay distinct, but both are {tokenizer.eos_token!r}. "
    "Set a distinct pad token in this model's profile."
)

print(f"model     : {MODEL_ID}")
print(f"pad token : {tokenizer.pad_token!r} (id {tokenizer.pad_token_id})")
print(f"eos token : {tokenizer.eos_token!r} (id {tokenizer.eos_token_id})")
print(f"bos token : {tokenizer.bos_token!r}")
print(f"vocab     : {len(tokenizer):,}")

## 3. The 4-bit configuration, argument by argument

```python
BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
```

**`load_in_4bit`** swaps every `nn.Linear` in the transformer blocks for a `bnb.nn.Linear4bit`.
Embeddings are `nn.Embedding`, not `Linear`, so they are untouched; the `lm_head` is skipped by
default as well. That is why the footprint is never `params ÷ 4` — section 4 measures the gap
rather than hand-waving it.

**`bnb_4bit_quant_type="nf4"`.** Weights in a trained transformer are roughly zero-centred and
roughly normal. NF4 — 4-bit NormalFloat — places its 16 representable values at the quantiles of a
normal distribution rather than spacing them evenly, so it spends resolution where the weights
actually are. Against plain `"fp4"` it is a free accuracy win at identical cost.

Quantization is **blockwise**: weights are cut into blocks (64 values), each block divided by its
own `absmax` before being mapped onto the 16 levels. One outlier therefore ruins the resolution of
its own 64 neighbours, not of the entire tensor.

**`bnb_4bit_use_double_quant=True`.** Those per-block `absmax` values are themselves fp32, and at
one per 64 weights that is 32/64 = **0.5 extra bits per weight** — 12% on top of a 4-bit budget.
Double quantization quantizes the absmax values too, to 8-bit with a second-level scale, taking the
overhead to roughly 0.13 bits per weight. Net cost of the whole scheme is about **4.13 bits per
weight** instead of 4.5.

**`bnb_4bit_compute_dtype=torch.bfloat16`.** Storage and arithmetic are different questions. The
weights sit in 4 bits, but every matmul dequantizes its block back to bf16 first and computes
there. Set this to fp32 and you pay for precision the 4-bit weights cannot supply; leave it at the
fp32 default and you silently do exactly that. This is the argument people most often omit.

Note what is *not* quantized: the LoRA adapters. They are created fresh in higher precision and
stay there, which is the entire trick — the frozen base is cheap, and the part receiving gradients
is not. `docs/concepts.md` §16 has the longer version.

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",               # quantiles of a normal, not an even grid
    bnb_4bit_use_double_quant=True,          # quantize the per-block absmax too
    bnb_4bit_compute_dtype=DTYPE,            # storage is 4-bit; every matmul still runs in bf16
)

# The bit budget, before we load anything - so the measurement in section 4 has something to
# be checked against rather than being taken on faith.
BLOCK = 64
for label, double in (("single quant", False), ("double quant", True)):
    absmax_bits = (8 / BLOCK + 32 / (BLOCK * 256)) if double else (32 / BLOCK)
    print(f"{label}: 4 + {absmax_bits:.3f} = {4 + absmax_bits:.3f} bits per quantized weight")
print("\n(the second term is the per-block absmax overhead; double quant stores it in 8 bits")
print(" with one fp32 scale per 256 blocks instead of fp32 per block)")

## 4. Load in 4-bit, and measure what it cost

Downloading 16 GB of fp16 weights takes a few minutes; they are quantized on the way onto the GPU,
so the 4-bit footprint is what ends up in VRAM.

Two things get measured here rather than asserted: the actual footprint against the fp16
counterfactual, and **which modules bitsandbytes actually converted**. The second one is where the
arithmetic usually stops matching people's expectations.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    **{DTYPE_KW: DTYPE},          # dtype of everything bnb did NOT quantize
)
model.config.pad_token_id = tokenizer.pad_token_id

print(f"\nloaded {MODEL_ID}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
def count_params(m):
    """Logical vs stored parameter counts.

    p.numel() lies on a 4-bit model: Params4bit packs two weights into each uint8 and
    reshapes to (n/2, 1), so summing numel() under-reports by ~2x on every quantized
    layer. Count Linear4bit by its logical shape instead.
    """
    logical, stored, quantized = 0, 0, 0
    accounted = set()
    for name, mod in m.named_modules():
        if isinstance(mod, bnb.nn.Linear4bit):
            n = mod.in_features * mod.out_features
            logical += n
            quantized += n
            stored += mod.weight.numel()
            accounted.add(f"{name}.weight")
            if mod.bias is not None:
                logical += mod.bias.numel()
                stored += mod.bias.numel()
                accounted.add(f"{name}.bias")
    for name, p in m.named_parameters():
        if name not in accounted:
            logical += p.numel()
            stored += p.numel()
    return logical, stored, quantized


N_PARAMS, N_STORED, N_QUANTIZED = count_params(model)
footprint = model.get_memory_footprint() / 1e9
fp16_would_be = N_PARAMS * 2 / 1e9

print(f"parameters (logical)      : {N_PARAMS/1e9:.2f} B")
print(f"  of which quantized      : {N_QUANTIZED/1e9:.2f} B  ({N_QUANTIZED/N_PARAMS:.1%})")
print(f"  left in {str(DTYPE).split('.')[-1]:9s}        : {(N_PARAMS-N_QUANTIZED)/1e9:.2f} B  "
      f"(embeddings + lm_head + norms)")
print(f"sum of p.numel()          : {N_STORED/1e9:.2f} B   <- what a naive count reports\n")

print(f"4-bit footprint           : {footprint:.2f} GB")
print(f"fp16 would have needed    : {fp16_would_be:.2f} GB   ({fp16_would_be/footprint:.1f}x)")
print(f"this GPU has              : {TOTAL_VRAM:.1f} GB\n")

headroom = TOTAL_VRAM - fp16_would_be
if headroom < 8:
    print(f"In fp16 this model leaves {headroom:.1f} GB for activations, gradients and optimizer")
    print("state on this GPU. That is not enough to train at this sequence length - which is")
    print("the entire reason QLoRA exists.")
else:
    print(f"In fp16 it would still fit here ({headroom:.1f} GB spare), but on a 24 GB card it")
    print("would not, and the memory saved is what pays for the larger model.")

In [ ]:
# Which modules did bitsandbytes actually convert? The footprint above is not params/4,
# and this is why.
kinds = Counter()
for name, mod in model.named_modules():
    if isinstance(mod, bnb.nn.Linear4bit):
        kinds["Linear4bit  (4-bit NF4)"] += 1
    elif isinstance(mod, torch.nn.Linear):
        kinds[f"Linear      ({str(mod.weight.dtype).split('.')[-1]})"] += 1
    elif isinstance(mod, torch.nn.Embedding):
        kinds[f"Embedding   ({str(mod.weight.dtype).split('.')[-1]})"] += 1

for kind, n in sorted(kinds.items()):
    print(f"  {n:>4d}  x  {kind}")

emb = model.get_input_embeddings().weight
print(f"\nembed_tokens: {tuple(emb.shape)} in {emb.dtype} = {emb.numel()*emb.element_size()/1e9:.2f} GB")
print("Embeddings are nn.Embedding, not nn.Linear, so bitsandbytes never sees them; the lm_head")
print("is skipped by default because quantizing the output projection costs real accuracy.")
print("On a large-vocabulary model those two tensors alone are most of what is left in bf16.")

## 5. Build the instruction data from PubMedQA

Every cell in this section, and in sections 6 to 8, is **byte-identical to notebook 02**. That is
not laziness — it is the precondition for the comparison at the end. Same articles, same split,
same seed, same prompt, same masking, so the only things that differ between the two notebooks'
accuracy numbers are model scale and quantization.

In [ ]:
from datasets import load_dataset

# --- constants mirrored in scripts/prepare_data.py and notebook 01 ---
DATASET = "qiaojin/PubMedQA"
SEED = 20260815
EVAL_FRACTION = 0.20      # -> 800 train / 200 eval

TASK = (
    "Answer the research question using only the abstract provided. "
    'Begin your reply with "Answer:" followed by yes, no, or maybe, '
    "then justify it in one or two sentences."
)

labeled = load_dataset(DATASET, "pqa_labeled", split="train")
print(labeled)
print("\nexpert decisions across all 1,000 records:",
      dict(Counter(labeled["final_decision"])))

In [ ]:
def join_sections(context) -> str:
    """Render an abstract's sections as `LABEL: text` blocks."""
    labels = context.get("labels") or []
    parts = []
    for i, text in enumerate(context["contexts"]):
        text = " ".join(text.split())
        if not text:
            continue
        label = (labels[i] if i < len(labels) else "").strip()
        parts.append(f"{label}: {text}" if label else text)
    return "\n\n".join(parts)


records = []
for row in labeled:
    abstract = join_sections(row["context"])
    if not abstract:
        continue
    records.append({
        "instruction": f'{TASK}\n\nQuestion: {row["question"].strip()}',
        "input": abstract,
        "output": f'Answer: {row["final_decision"]}\n\n{" ".join(row["long_answer"].split())}',
        "decision": row["final_decision"],
        "pubid": row["pubid"],
    })

print(f"{len(records)} instruction records built")
print("\n--- one rendered record ---")
r = records[0]
print(f"INSTRUCTION:\n{r['instruction']}\n")
print(f"INPUT:\n{r['input'][:400]} ...\n")
print(f"OUTPUT:\n{r['output'][:300]}")

### Stratified split, and a leakage check

`maybe` is only 11% of the data. An unstratified split would leave the eval set with an unstable
number of them and make accuracy noisy, so we split within each decision class.

We also assert that no article appears in both stages. If stage 1 had already read these abstracts,
stage 2's accuracy would be measuring memorisation.

In [ ]:
rng = random.Random(SEED + 1)
by_decision = {}
for r in records:
    by_decision.setdefault(r["decision"], []).append(r)

train_records, eval_records = [], []
for decision in sorted(by_decision):
    rows = sorted(by_decision[decision], key=lambda r: r["pubid"])
    rng.shuffle(rows)
    n_eval = round(len(rows) * EVAL_FRACTION)
    eval_records.extend(rows[:n_eval])
    train_records.extend(rows[n_eval:])
rng.shuffle(train_records); rng.shuffle(eval_records)

print(f"train {len(train_records)}  {dict(Counter(r['decision'] for r in train_records))}")
print(f"eval  {len(eval_records)}  {dict(Counter(r['decision'] for r in eval_records))}")

eval_counts = Counter(r["decision"] for r in eval_records)
MAJORITY_LABEL, majority_n = eval_counts.most_common(1)[0]
MAJORITY_BASELINE = majority_n / len(eval_records)
print(f"\nmajority-class baseline: {MAJORITY_BASELINE:.1%} (always answer '{MAJORITY_LABEL}')")
print("Any model that cannot beat that number has learned nothing useful.")

In [ ]:
# Article-level disjointness between the two stages. pqa_unlabeled (stage 1) and
# pqa_labeled (stage 2) cover different papers, but assert it rather than trust it.
unlabeled_ids = set(load_dataset(DATASET, "pqa_unlabeled", split="train")["pubid"])
stage2_ids = {r["pubid"] for r in records}
overlap = unlabeled_ids & stage2_ids
print(f"pqa_unlabeled articles : {len(unlabeled_ids):,}")
print(f"pqa_labeled articles   : {len(stage2_ids):,}")
print(f"overlap                : {len(overlap)}")
assert not overlap, f"{len(overlap)} articles appear in both stages — stage 2 eval is contaminated"
print("\nNo article is shared between the stages.")

## 6. The prompt template, unchanged

Both base models here are pretrained-only, so neither ships a chat template — there is no
`<|im_start|>`, no `[INST]`, nothing to apply. The template below is the same Alpaca-style pair
notebooks 02 and 03 use, kept identical so the held-out records are presented to the 8B model in
exactly the form the 1B saw.

In [ ]:
PROMPT_WITH_INPUT = (
    "Below is an instruction describing a task, paired with input providing further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)

PROMPT_NO_INPUT = (
    "Below is an instruction describing a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Response:\n"
)


def build_prompt(record) -> str:
    """The part the model reads but is NOT trained to produce."""
    template = PROMPT_WITH_INPUT if record["input"].strip() else PROMPT_NO_INPUT
    return template.format(instruction=record["instruction"].strip(),
                           input=record["input"].strip())


print(build_prompt(train_records[0])[:700] + " ...")
print("\n>>> TARGET >>>")
print(train_records[0]["output"] + tokenizer.eos_token)

## 7. Completion-only loss masking

Unchanged from notebook 02 except that `MAX_LEN` now comes from the model profile, because the
Qwen tokenizer segments this text differently and the budget has to be re-measured rather than
assumed. The assert is the same one: a truncated target teaches the model to stop mid-sentence.

In [ ]:
MAX_LEN = PROFILE["max_len"]   # re-measured below; the assert at the end is the real check


def tokenize_record(record):
    """Tokenize one record, masking the prompt out of the loss."""
    prompt = build_prompt(record)
    answer = record["output"].strip()

    # add_special_tokens=True puts BOS at the front of the prompt only.
    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
    answer_ids = answer_ids + [tokenizer.eos_token_id]   # <-- teaches the model to stop

    input_ids = (prompt_ids + answer_ids)[:MAX_LEN]
    # -100 over the prompt span, real ids over the answer span.
    labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_LEN]

    return {"input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels}


train_tok = [tokenize_record(r) for r in train_records]
eval_tok = [tokenize_record(r) for r in eval_records]

supervised = sum(sum(1 for t in r["labels"] if t != -100) for r in train_tok)
total = sum(len(r["labels"]) for r in train_tok)
lengths = sorted(len(r["input_ids"]) for r in train_tok)

print(f"sequence length: p50={lengths[len(lengths)//2]} p90={lengths[int(len(lengths)*.9)]} max={lengths[-1]}")
print(f"supervised tokens: {supervised:,} / {total:,} = {supervised/total:.1%}")
print(f"\nWithout the mask, {1 - supervised/total:.0%} of the training signal would go on")
print("reproducing abstracts the model is only supposed to READ.")

truncated = sum(1 for r in train_tok + eval_tok if len(r["input_ids"]) >= MAX_LEN)
assert truncated == 0, f"{truncated} records hit MAX_LEN — a truncated target teaches the model to stop mid-sentence"
print(f"\nrecords truncated at MAX_LEN={MAX_LEN}: {truncated}")

### Sanity check: show exactly which tokens are supervised

Don't take the masking on trust. Decode it.

In [ ]:
sample = train_tok[0]
kept = [t for t in sample["labels"] if t != -100]
masked_n = len(sample["labels"]) - len(kept)

print(f"total tokens  : {len(sample['input_ids'])}")
print(f"masked (-100) : {masked_n}")
print(f"supervised    : {len(kept)}\n")
print("--- MASKED, tail of it (read, not scored) " + "-" * 38)
print(tokenizer.decode(sample["input_ids"][max(0, masked_n - 60):masked_n]))
print("\n--- SUPERVISED (the entire training signal) " + "-" * 36)
print(tokenizer.decode(kept))
print(f"\nNote the trailing {tokenizer.eos_token}: that is the EOS the model learns to emit.")

## 8. Collator

Verbatim from notebook 02. Three pad values, three meanings: `pad_token_id` so the ids stay valid,
`0` so attention ignores the padding, `-100` so the loss does.

In [ ]:
from datasets import Dataset


def collate(features):
    """Dynamic padding. Note the three different pad values."""
    longest = max(len(f["input_ids"]) for f in features)

    def pad(seq, value):
        return seq + [value] * (longest - len(seq))

    return {
        "input_ids": torch.tensor([pad(f["input_ids"], tokenizer.pad_token_id) for f in features]),
        "attention_mask": torch.tensor([pad(f["attention_mask"], 0) for f in features]),
        "labels": torch.tensor([pad(f["labels"], -100) for f in features]),
    }


train_ds = Dataset.from_list(train_tok)
eval_ds = Dataset.from_list(eval_tok)

demo = collate([train_tok[0], train_tok[1]])
print({k: tuple(v.shape) for k, v in demo.items()})
shorter = 0 if len(train_tok[0]["input_ids"]) < len(train_tok[1]["input_ids"]) else 1
print(f"\nlast label of the shorter row: {demo['labels'][shorter][-1].item()}  (-100 if it was padded)")

## 9. `prepare_model_for_kbit_training`, and what it actually does

Notebooks 01 and 02 do part of this by hand:

```python
for name, param in model.named_parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()
```

`docs/concepts.md` §8 already describes that loop as "what `prepare_model_for_kbit_training` does
for QLoRA". Here is the real thing, and it does more than the loop:

1. **Freezes every base parameter.** The 4-bit weights are not trainable and never will be —
   there is no meaningful gradient for a value stored on a 16-level grid.
2. **Upcasts the remaining fp16/bf16 parameters to fp32.** Layernorms are the ones that matter:
   they are precision-sensitive and cheap. On a large-vocabulary model this also catches the
   embeddings and the `lm_head`, which are *not* cheap — the cell below measures the cost instead
   of pretending it is free.
3. **Wires gradient checkpointing**, so activations are recomputed in the backward pass rather
   than stored.

**The gotcha, named.** Gradient checkpointing plus PEFT needs `enable_input_require_grads()`. The
frozen embedding layer produces outputs that do not require grad, checkpointing has nothing to
attach a graph to, and the first backward dies with:

```
RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn
```

It is one line, it is easy to omit, and the error names nothing you touched.
`prepare_model_for_kbit_training` already calls it for you when
`use_gradient_checkpointing=True` — which is the better argument for using the helper than for
hand-rolling the upcast loop. The call below is left in explicitly because the failure it prevents
is worth being able to recognise.

In [ ]:
from peft import prepare_model_for_kbit_training

before_dtypes = Counter(str(p.dtype) for p in model.parameters())
before_mem = model.get_memory_footprint() / 1e9

model.config.use_cache = False           # incompatible with gradient checkpointing
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model.enable_input_require_grads()       # <-- without this the first backward has no graph

after_dtypes = Counter(str(p.dtype) for p in model.parameters())
after_mem = model.get_memory_footprint() / 1e9

print("parameter tensors by dtype")
print(f"{'dtype':>16s}{'before':>10s}{'after':>10s}")
for d in sorted(set(before_dtypes) | set(after_dtypes)):
    print(f"{d:>16s}{before_dtypes.get(d, 0):>10d}{after_dtypes.get(d, 0):>10d}")

print(f"\nfootprint: {before_mem:.2f} GB -> {after_mem:.2f} GB  ({after_mem - before_mem:+.2f} GB)")
print("The increase is the fp32 upcast of the embeddings, the lm_head and the norms - the")
print("tensors bitsandbytes left alone. It is the price of numerically stable training, and it")
print("is why a '4-bit 8B' does not occupy 4 GB in practice.")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\ntrainable parameters right now: {trainable:,}  (LoRA has not been attached yet)")

## 10. Attach LoRA and train

The `LoraConfig` is **identical** to the one in all three earlier notebooks — same `r`, same
`lora_alpha`, same seven `target_modules`. Holding it fixed is deliberate: the accuracy difference
at the end should be attributable to scale and quantization, not to a rank change smuggled in
alongside them.

`r=16` on an 8B model is about 42M trainable parameters, roughly 0.5% of the total. The QLoRA paper
uses `r=64`; that is a reasonable thing to try afterwards, one line away, and is listed in section
17.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# The adapters are NOT quantized. That is the whole trick: the frozen base is cheap, and the
# part that receives gradients keeps the precision gradients need.
a_sample = model.base_model.model.model.layers[PROFILE["probe_layer"]].mlp.down_proj
_qs = a_sample.base_layer.weight.quant_state
print(f"\nbase layer  : {type(a_sample.base_layer).__name__}")
print(f"   stored as : {a_sample.base_layer.weight.dtype} on a {_qs.quant_type} grid "
      f"(logical dtype {_qs.dtype})")
print(f"lora_A      : {type(a_sample.lora_A['default']).__name__}  "
      f"weight dtype {a_sample.lora_A['default'].weight.dtype}  <- not quantized")
print(f"scaling     : alpha/r = {a_sample.scaling['default']}")

### The optimizer, honestly

`optim="paged_adamw_8bit"` is the QLoRA default and it does two separate things that are worth
keeping apart.

**8-bit state** halves-and-halves-again the optimizer memory: AdamW holds two moments per trainable
parameter, so at ~42M parameters that is ~336 MB in fp32 against ~84 MB in 8-bit. A real saving,
but not a decisive one — with LoRA the trainable set is tiny by construction. It matters far more
when the thing being trained is the *whole* model.

**Paging** is the part that earns its place here. The optimizer state lives in unified memory, so a
transient spike — a long batch, a fragmented allocator — spills to host RAM instead of raising
`CUDA out of memory` forty minutes into a run. That is cheap insurance on rented hardware.

The cell below prints both numbers so the claim can be checked rather than believed.

In [ ]:
from transformers import Trainer, TrainingArguments

OUT_DIR = f"/content/outputs/qlora-{MODEL_SHORT}-instruct"

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,      # effective batch 8 - same as notebook 02
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="paged_adamw_8bit",           # bitsandbytes: 8-bit moments, unified-memory paging
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

# On an A100 40GB, per_device_train_batch_size=4 with gradient_accumulation_steps=2 keeps the
# effective batch at 8 and roughly halves the wall clock. Left at 2/4 so an L4 runs it too.

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collate,
)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
steps = len(train_ds) // (args.per_device_train_batch_size * args.gradient_accumulation_steps)
print(f"trainable parameters : {n_trainable:,}  ({n_trainable/N_PARAMS:.2%} of the model)")
print(f"AdamW state, fp32    : {n_trainable * 8 / 1e6:.0f} MB")
print(f"AdamW state, 8-bit   : {n_trainable * 2 / 1e6:.0f} MB   "
      f"(saves {n_trainable * 6 / 1e6:.0f} MB)")
print(f"optimizer steps      : ~{steps * args.num_train_epochs}\n")

torch.cuda.reset_peak_memory_stats()
trainer.train()

In [ ]:
peak = torch.cuda.max_memory_allocated() / 1e9
print(f"peak GPU memory during training : {peak:.2f} GB of {TOTAL_VRAM:.1f} GB")
print(f"  4-bit weights                 : ~{after_mem:.2f} GB")
print(f"  everything else               : ~{peak - after_mem:.2f} GB  "
      "(activations, gradients, optimizer state)")
print(f"\nfp16 weights alone would have been {fp16_would_be:.2f} GB, before any of that.")

## 11. Loss curves

Loss on response tokens only, since the prompt span is masked to `-100`. Absolute values are not
comparable to notebook 02's — a different tokenizer and a different model give a different
per-token scale. The shape is what to read.

In [ ]:
history = trainer.state.log_history
train_pts = [(h["epoch"], h["loss"]) for h in history if "loss" in h]
eval_pts = [(h["epoch"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(9, 4))
plt.plot(*zip(*train_pts), label="train loss", color="#4C72B0", alpha=0.8)
if eval_pts:
    plt.plot(*zip(*eval_pts), label="eval loss (held-out records)", color="#C44E52", marker="o")
plt.xlabel("epoch"); plt.ylabel("loss (response tokens only)"); plt.legend()
plt.title(f"QLoRA: 4-bit instruction tuning on PubMedQA ({MODEL_ID})"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

if eval_pts:
    print(f"eval loss: {eval_pts[0][1]:.3f} (epoch {eval_pts[0][0]:.0f}) "
          f"-> {eval_pts[-1][1]:.3f} (epoch {eval_pts[-1][0]:.0f})")

## 12. Save the adapter — before anything merges

**Order matters from here on.** Section 14 merges the adapter into the 4-bit weights, and that is
destructive: it rewrites the base tensors in place and unwraps the PEFT model. Saving afterwards
would save the damaged artifact.

What gets saved is the adapter alone — roughly 160 MB against 16 GB of base weights. The base is
downloaded from the Hub by whoever loads it; the adapter is the only thing this run produced.

In [ ]:
import shutil

ADAPTER_DIR = Path(OUT_DIR)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
size_mb = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file()) / 1e6
print(f"Saved to {ADAPTER_DIR}  ({size_mb:.1f} MB)")
print(f"base model on the Hub: {fp16_would_be:.1f} GB — the adapter is "
      f"{size_mb / (fp16_would_be * 1000):.2%} of it")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path(f"/content/drive/MyDrive/finetuning-demo/qlora-{MODEL_SHORT}-instruct")
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(ADAPTER_DIR, dest)
    print(f"Copied to {dest}")
except ImportError:
    print("Not running in Colab — adapter is on local disk only.")

## 13. Decision accuracy on the same 200 held-out records

The grading regex, the generation helper and the scorer are **byte-identical to notebook 02**, so
the numbers land on the same scale as the ones that notebook produced. Greedy decoding throughout,
so a re-run gives the same answers.

In [ ]:
DECISION_RE = re.compile(r"answer\s*:?\s*\b(yes|no|maybe)\b", re.IGNORECASE)


@torch.no_grad()
def generate_batch(m, batch, max_new_tokens=120):
    """Greedy, left-padded batched generation. Deterministic so runs are comparable."""
    m.eval()
    m.config.use_cache = True
    prompts = [build_prompt(r) for r in batch]
    # Left padding for generation: every sequence then ends at the same position,
    # which is where the model continues from. (Training used right padding.)
    tokenizer.padding_side = "left"
    enc = tokenizer(prompts, return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_LEN).to("cuda")
    out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                     eos_token_id=tokenizer.eos_token_id,
                     pad_token_id=tokenizer.pad_token_id)
    tokenizer.padding_side = "right"
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen]


def score(m, dataset, label, batch_size=8):
    """Decision accuracy + parse rate over the held-out set."""
    texts = []
    for i in range(0, len(dataset), batch_size):
        texts.extend(generate_batch(m, dataset[i:i + batch_size]))
        print(f"\r  {label}: {min(i + batch_size, len(dataset))}/{len(dataset)}", end="")
    print()
    preds = []
    for t in texts:
        hit = DECISION_RE.search(t)
        preds.append(hit.group(1).lower() if hit else None)
    correct = sum(1 for p, r in zip(preds, dataset) if p == r["decision"])
    parsed = sum(1 for p in preds if p is not None)
    return {"accuracy": correct / len(dataset), "parse_rate": parsed / len(dataset),
            "preds": preds, "texts": texts}

### The order these are measured in is not arbitrary

Three systems, one model in memory, no reloads — but only if they are measured in the right order:

1. **4-bit base**, inside `with model.disable_adapter():`. Switching the adapter off is free, and
   it is meaningless once the adapter has been merged in.
2. **+ QLoRA**, the model as trained.
3. **+ QLoRA, merged** — in section 14, *after* the merge, which is destructive.

Get this order wrong and you either reload an 8B model twice or measure the wrong thing. Expect
the base row's parse rate to be near zero: it has never been taught to emit `Answer:`, exactly as
in notebook 02.

In [ ]:
# Scored on the full 200-record held-out set. A few minutes per system on an A100.
model.config.use_cache = True     # generation wants the KV cache back
results = {}

with model.disable_adapter():
    results["4-bit base"] = score(model, eval_records, "4-bit base")

results["4-bit + qlora"] = score(model, eval_records, "4-bit + qlora")

gold = [r["decision"] for r in eval_records]
print("\ndone with the unmerged systems")

In [ ]:
# Notebook 02's stage-2 result, for scale context. Paste the accuracy it printed here.
# It is a CROSS-SESSION number: different model, different session, and that pipeline had a
# domain-adaptation stage this notebook does not. Context, not a controlled comparison.
STAGE2_1B_ACCURACY = None      # e.g. 0.585

print(f"Held-out set: {len(eval_records)} records   {dict(Counter(gold))}\n")
print(f"{'system':30s}{'accuracy':>10s}{'parse rate':>13s}")
print("-" * 53)
print(f"{'majority class (' + MAJORITY_LABEL + ')':30s}{MAJORITY_BASELINE:>9.1%}{'n/a':>13s}")
for name in ["4-bit base", "4-bit + qlora"]:
    r = results[name]
    print(f"{name:30s}{r['accuracy']:>9.1%}{r['parse_rate']:>12.1%}")
print("-" * 53)

final = results["4-bit + qlora"]["accuracy"]
print(f"\nvs majority class : {final - MAJORITY_BASELINE:+.1%}")
print(f"vs 4-bit base     : {final - results['4-bit base']['accuracy']:+.1%}")
if STAGE2_1B_ACCURACY is not None:
    print(f"vs 1B stage 2     : {final - STAGE2_1B_ACCURACY:+.1%}  "
          "(cross-session, and that model had domain adaptation first)")
else:
    print("\nSet STAGE2_1B_ACCURACY from notebook 02's output for the scale comparison.")

In [ ]:
# Confusion matrix for the final model — is it actually discriminating, or just saying "yes"?
LABELS = ["yes", "no", "maybe"]
preds = results["4-bit + qlora"]["preds"]

print("rows = expert label, cols = prediction\n")
print(f"{'':8s}" + "".join(f"{c:>8s}" for c in LABELS) + f"{'unparsed':>10s}")
for actual in LABELS:
    row = [sum(1 for p, g in zip(preds, gold) if g == actual and p == c) for c in LABELS]
    unparsed = sum(1 for p, g in zip(preds, gold) if g == actual and p is None)
    print(f"{actual:8s}" + "".join(f"{v:>8d}" for v in row) + f"{unparsed:>10d}")

print(f"\nprediction distribution: {dict(Counter(p or 'unparsed' for p in preds))}")
print(f"expert distribution    : {dict(Counter(gold))}")

## 14. The merge constraint, measured

This repo has been making a claim since `docs/concepts.md` §3: you cannot cleanly merge a LoRA
update into quantized weights, and that is why notebooks 01-03 keep their base in fp16. The claim
is correct. It has also, until now, been unquantified.

Here is the mechanism. Merging means computing $W' = W + \frac{\alpha}{r}BA$ and storing the
result — but the store has to go back onto the 4-bit grid. So what actually gets kept is
$Q^{-1}(Q(W'))$, and the requantization error $\varepsilon = Q^{-1}(Q(W')) - W'$ is noise added on
top of the very update you spent an hour training.

The number that decides whether this matters is not $\|\varepsilon\|$ on its own. It is the ratio
$\|\varepsilon\| / \|\Delta W\|$ — the size of the noise **relative to the size of the update**.
If that lands near or above 1, merging discards roughly as much as training added.

Measure it on one weight first, before touching the model.

In [ ]:
PROBE_LAYER = PROFILE["probe_layer"]
probe = model.base_model.model.model.layers[PROBE_LAYER].mlp.down_proj
qw = probe.base_layer.weight

# What the 4-bit weights currently hold, back in bf16.
W = bnb.functional.dequantize_4bit(qw.data, qw.quant_state).float()

# What training actually learned, at the scale it is applied with.
A_ = probe.lora_A["default"].weight.float()
B_ = probe.lora_B["default"].weight.float()
dW = probe.scaling["default"] * (B_ @ A_)
assert dW.shape == W.shape, f"shape mismatch: {dW.shape} vs {W.shape}"

# What a merge would have to store, and what survives the round trip.
Wm = (W + dW).to(qw.quant_state.dtype).contiguous()
q, st = bnb.functional.quantize_4bit(
    Wm,
    blocksize=qw.quant_state.blocksize,
    compress_statistics=qw.quant_state.nested,
    quant_type=qw.quant_state.quant_type,
)
err = (bnb.functional.dequantize_4bit(q, st).float() - Wm.float())

nW, ndW, nerr = W.norm().item(), dW.norm().item(), err.norm().item()

print(f"probe: model.layers.{PROBE_LAYER}.mlp.down_proj   {tuple(W.shape)}")
print(f"  quant type / blocksize / nested : {st.quant_type} / {st.blocksize} / {st.nested}\n")
print(f"  ||W||                   : {nW:11.4f}   the quantized base weight")
print(f"  ||dW||                  : {ndW:11.4f}   what LoRA learned")
print(f"  ||requant error||       : {nerr:11.4f}   what the merge would throw away\n")
print(f"  ||dW|| / ||W||          : {ndW / nW:11.4%}   the update is a small perturbation")
print(f"  ||err|| / ||W||         : {nerr / nW:11.4%}   so is the noise")
print(f"  ||err|| / ||dW||        : {nerr / ndW:11.3f}   <- the number that matters")

print()
if nerr / ndW >= 1.0:
    print("  The requantization noise is at least as large as the update. Merging here does not")
    print("  approximate the fine-tuned model; it largely overwrites it with rounding error.")
elif nerr / ndW >= 0.3:
    print("  The noise is a substantial fraction of the update. Merging is lossy in a way that")
    print("  should be measured on the task, not waved through.")
else:
    print("  The noise is small relative to the update on this tensor. That is a per-tensor")
    print("  result, not a licence - check the task metric below before trusting it.")

### Now the end-to-end version

A common expectation is that `merge_and_unload()` simply refuses on a 4-bit model. **It does
not.** PEFT supports merging into `Linear4bit`: it dequantizes, adds the update, and requantizes —
emitting a warning about rounding rather than an error. Silent-ish lossiness is a worse failure
mode than a refusal would be, because nothing stops you shipping the result.

The cell below reports whatever actually happens under the installed PEFT version, then re-scores.
The merged weights are deliberately **not** saved: the adapter from section 12 is the artifact
worth keeping, and section 15 has the export path that does not lose anything.

In [ ]:
import warnings

merged_ok = False
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    try:
        model = model.merge_and_unload()
        merged_ok = True
        print(f"merge_and_unload() returned {type(model).__name__}")
    except Exception as exc:
        print(f"merge_and_unload() raised {type(exc).__name__}: {exc}")
        print("\nThat is a legitimate outcome too - record it and skip the merged row.")

for w in caught:
    print(f"  [{w.category.__name__}] {w.message}")

if merged_ok:
    still_4bit = sum(1 for m in model.modules() if isinstance(m, bnb.nn.Linear4bit))
    print(f"\nLinear4bit modules after the merge: {still_4bit}")
    print("The base is still 4-bit. The update was folded in and requantized onto the same grid.")
    model.config.use_cache = True
    results["4-bit + qlora, merged"] = score(model, eval_records, "merged")

In [ ]:
if merged_ok:
    p_unmerged = results["4-bit + qlora"]["preds"]
    p_merged = results["4-bit + qlora, merged"]["preds"]

    print(f"{'system':30s}{'accuracy':>10s}{'parse rate':>13s}")
    print("-" * 53)
    print(f"{'majority class (' + MAJORITY_LABEL + ')':30s}{MAJORITY_BASELINE:>9.1%}{'n/a':>13s}")
    for name in ["4-bit base", "4-bit + qlora", "4-bit + qlora, merged"]:
        r = results[name]
        print(f"{name:30s}{r['accuracy']:>9.1%}{r['parse_rate']:>12.1%}")
    print("-" * 53)

    drift = results["4-bit + qlora, merged"]["accuracy"] - results["4-bit + qlora"]["accuracy"]
    changed = sum(1 for a, b in zip(p_unmerged, p_merged) if a != b)
    print(f"\nmerged vs unmerged     : {drift:+.1%}")
    print(f"predictions changed    : {changed} / {len(gold)}")

    # McNemar, exact binomial on the discordant pairs only. Same test notebook 03 uses:
    # the two systems answered the SAME 200 questions, so the comparison is paired.
    b = sum(1 for a, c, g in zip(p_unmerged, p_merged, gold) if a == g and c != g)
    c_ = sum(1 for a, c, g in zip(p_unmerged, p_merged, gold) if a != g and c == g)
    n = b + c_
    p_value = 1.0
    if n:
        lo = min(b, c_)
        p_value = min(1.0, 2 * sum(math.comb(n, k) for k in range(lo + 1)) / 2 ** n)

    print(f"\nMcNemar, paired on the same {len(gold)} records")
    print(f"  agree                        : {len(gold) - n}")
    print(f"  unmerged right, merged wrong : {b}")
    print(f"  unmerged wrong, merged right : {c_}")
    print(f"  two-sided exact p            : {p_value:.3f}")
    print("\n  " + ("No detectable damage at n=200. Note that 'not detectable here' is not"
                    "\n  'harmless' - the weight-level error above is real either way, and a"
                    "\n  200-record test cannot see a small degradation."
                    if p_value >= 0.05 else
                    "The merge changed predictions lopsidedly enough to be unlikely under"
                    "\n  no true difference. This is the constraint, showing up on the task metric."))
else:
    print("Merge did not complete - nothing to compare.")

## 15. The export path that is actually correct

If you want a single merged checkpoint to serve, do the merge **at full precision**, not on the
4-bit copy: load the base in bf16 on CPU, apply the saved adapter, `merge_and_unload()` there, and
save. Nothing is requantized, so nothing is lost — and you can quantize the *result* afterwards if
you want, which is a different and much better-behaved operation than quantizing mid-merge.

The cost is host RAM: an 8B model in bf16 is ~16 GB, and Colab's standard runtime has about
12.7 GB. So this needs a high-RAM runtime or a machine outside Colab.

It is off by default — flip `RUN_FP16_EXPORT` if you have the RAM and want the artifact. It
re-downloads the base weights, so budget a few minutes.

In [ ]:
RUN_FP16_EXPORT = False       # needs ~2x the model size in host RAM

if RUN_FP16_EXPORT:
    import psutil

    ram_gb = psutil.virtual_memory().total / 1e9
    need = fp16_would_be * 2      # the loaded model, plus room for the merged copy
    print(f"host RAM: {ram_gb:.1f} GB | need ~{need:.1f} GB")

    if ram_gb < need:
        print(f"\nSkipping: not enough host RAM. Use a high-RAM runtime, or run this step on a")
        print("machine with more memory - the adapter in section 12 is all you need to do it.")
    else:
        del model, trainer
        gc.collect()
        torch.cuda.empty_cache()

        from peft import PeftModel

        base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, device_map="cpu", **{DTYPE_KW: DTYPE},
        )
        probe_name = f"model.layers.{PROBE_LAYER}.mlp.down_proj.weight"
        pre = dict(base.named_parameters())[probe_name].detach().clone()

        merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
        post = dict(merged.named_parameters())[probe_name].detach()

        assert not torch.equal(pre, post), (
            "The merge did not change any weights. The adapter was not applied."
        )
        print(f"\nmerged in {DTYPE} on CPU; max |delta| on {probe_name}: "
              f"{(post.float() - pre.float()).abs().max().item():.6f}")

        export = Path(f"/content/outputs/qlora-{MODEL_SHORT}-merged-bf16")
        merged.save_pretrained(export)
        tokenizer.save_pretrained(export)
        print(f"saved to {export}")
else:
    print("RUN_FP16_EXPORT is off. The adapter from section 12 is the artifact worth keeping;")
    print("this step only matters if you need one self-contained checkpoint to serve.")

## 16. Qualitative side-by-side

The numbers say whether it is right. These say whether it is answering the question at all.

In [ ]:
SHOWN = [k for k in ("4-bit base", "4-bit + qlora", "4-bit + qlora, merged") if k in results]


def show(text, limit=420):
    text = " ".join(text.split())
    return text if len(text) <= limit else text[:limit] + " ..."


for i in range(4):
    r = eval_records[i]
    print("=" * 112)
    print(f"Q: {r['instruction'].split('Question: ')[-1]}")
    print(f"   [abstract] {show(r['input'], 180)}\n")
    for label in SHOWN:
        print(f"  {label:22s}: {show(results[label]['texts'][i])}\n")
    print(f"  {'EXPERT':22s}: {show(r['output'])}")
print("=" * 112)

In [ ]:
# How long does each system run before it stops? A tuned model should stop on its own.
print("".join(f"{k:>24s}" for k in SHOWN))
print("-" * (24 * len(SHOWN)))
for i in range(min(8, len(eval_records))):
    lens = [len(tokenizer(results[k]["texts"][i], add_special_tokens=False)["input_ids"])
            for k in SHOWN]
    print("".join(f"{n:>24d}" for n in lens))

avg = {k: sum(len(tokenizer(t, add_special_tokens=False)["input_ids"])
              for t in results[k]["texts"]) / len(eval_records)
       for k in SHOWN}
print("\nmean generated length: " + ", ".join(f"{k} {v:.0f}" for k, v in avg.items()))
print("A mean well under 120 means the model learned to emit EOS rather than run to the cap.")

## 17. What you built

```
unsloth/Meta-Llama-3.1-8B  (base, no instruction tuning)
      │   16.1 GB fp16  ->  ~5.7 GB in 4-bit NF4, double-quantized
      │
      └─ one LoRA, r=16, ~42M trainable params (~0.5%), on 800 PubMedQA records
           │
           ├─ scored against the 4-bit base on the same 200 held-out articles
           └─ merged into the 4-bit weights, and re-scored, to price the merge
```

An 8B model instruction-tuned on one GPU, with a ~160 MB artifact at the end of it.

### The four things this notebook is built to show

1. **4-bit is a storage decision, not an arithmetic one.** `bnb_4bit_compute_dtype` keeps every
   matmul in bf16. Weights on a 16-level grid, arithmetic at full width.
2. **The footprint is never `params ÷ 4`.** Embeddings and the `lm_head` stay unquantized, and
   `prepare_model_for_kbit_training` then upcasts them to fp32. Section 4 and section 9 measure
   both, because the gap between the advertised number and the real one is where OOMs come from.
3. **`p.numel()` under-reports on a quantized model.** `Params4bit` packs two weights per byte, so
   a naive parameter count halves. Section 4 counts by logical shape instead.
4. **Merging into 4-bit is permitted, lossy, and quiet.** PEFT does not refuse. Section 14 puts a
   ratio on it — requantization noise against the size of the trained update — and then checks
   whether it shows up on the task at all.

### Honest limitations

- **This is not a like-for-like replacement for notebooks 01-03.** It is one stage, not three.
  The 1B pipeline had domain adaptation before instruction tuning; this model did not. Any
  comparison against notebook 02's number is confounded by that as well as by scale, and
  `STAGE2_1B_ACCURACY` is a cross-session reference, not a controlled result.
- **A 200-record accuracy difference under about 7 points is noise**, exactly as in the rest of
  this repo. That applies to the 8B-vs-1B gap as much as to the merge drift, and only the merge
  comparison here is paired well enough for McNemar to help.
- **The quantization tax is not isolated.** Comparing a 4-bit 8B against an fp16 1B varies two
  things at once. Separating them needs the same model scored in both precisions — cheap at 1B,
  and listed below.
- **800 training examples is still very small.** The 8B model brings more pretrained knowledge to
  the task, not more supervision.
- **One `r`, one learning rate, no sweep.** `r=16` was held fixed for comparability with the
  earlier notebooks, not because it is optimal for 8B — the QLoRA paper uses `r=64`.
- **Nothing here is medical advice.** This is a training-mechanics exercise.

### Where to go next

| Change | Why |
|---|---|
| **Isolate the quantization tax** | Score `Llama-3.2-1B` + the stage-2 adapter in fp16 and again in NF4. Same model, same adapter, only the precision differs — the one clean measurement of what 4-bit costs. |
| **`r=64`, `lora_alpha=16`** | The QLoRA paper's configuration. `r=16` here was chosen to match the earlier notebooks, not to win. |
| **`Qwen/Qwen2.5-7B`** | Flip `MODEL_KEY`. Different tokenizer, different pad-token trap, same pipeline — a good test of whether the code was actually general. |
| **8-bit instead of 4-bit** | `load_in_8bit=True` costs about twice the memory and loses far less. Worth measuring rather than assuming 4-bit is always the right trade. |
| **QLoRA + DPO** | Notebook 03's stage at this scale. The reference policy is a second adapter, not a second model, so it is more affordable than it sounds. |
| **Unsloth** | ~2x faster and lower memory for exactly this workload, with the same PEFT artifacts. |
| **`pqa_artificial`** | 211k auto-labelled records. Still the highest-return change in this repo, at any model size. |
| **Serve the merged bf16 export** | Section 15 produces it. Quantize *after* merging, not during — with GPTQ or AWQ, which calibrate on data rather than rounding blindly. |

See `docs/concepts.md` §16 for the reasoning behind the quantization choices here, and §3 for why
notebooks 01-03 deliberately do not make them.